# bridge_player_playstyle.py

## Purpose
Bridge table that links players to all their PlayStyles (base + plus).

## Sources
- Silver: `fifa.silver.dim_player`, `fifa.silver.dim_playstyle`

## Business Rules
- A player may have 0..N PlayStyles (comma-separated string)
- Each PlayStyle can be base or plus
- Output: one row per (`player_snapshot_key`, `playstyle_id`, `is_plus`)

## Strategy
- Overwrite mode (small table, rebuilt on every run)
- Explodes the `playstyles` string, detects the `+` suffix, maps via `dim_playstyle`

## Output
- Table: `fifa.silver.bridge_player_playstyle`

## Columns
- `player_snapshot_key` (STRING)
- `player_id` (INT)
- `playstyle_id` (INT)
- `is_plus` (BOOLEAN)

## Why a bridge table?

A player can have multiple PlayStyles, and a PlayStyle can be assigned to many players. That's a **many-to-many** relationship.

In a star schema, M:N relationships are resolved with a **bridge table**:

dim_playstyle 

▼

bridge_player_playstyle 

▼

dim_player

## How the join works

The join between the exploded PlayStyles and `dim_playstyle` uses **two columns** as the key:

```python
on=["playstyle_name", "is_plus"]
```

This is required because "Finesse Shot" (base) and "Finesse Shot+" (plus) share the same playstyle_name but differ in is_plus.


## Why player_snapshot_key?
The bridge table links to dim_player via player_snapshot_key (not just player_id) because:

player_id alone is not unique in dim_player (a player has one row per edition).

The relationship in Power BI needs a unique key.

## Important: always keep player_snapshot_key
In every .select() after the explode, player_snapshot_key must be preserved. Otherwise, the link to dim_player is lost.

In [0]:
from pyspark.sql.functions import col, current_timestamp, to_date, lit, regexp_extract, when, concat,trim, explode, split, pmod, xxhash64

# --- Load dimensions ---
dim_player = spark.read.table("fifa.silver.dim_player").filter(col("Is_current") == True)
dim_playstyle = spark.read.table("fifa.silver.dim_playstyle").filter(col("Is_current") == True)

# --- Explode playstyles ---
bridge = (
    dim_player
    .filter(col("playstyles").isNotNull() & (col("playstyles") != ""))
    .select(
        col("player_snapshot_key"),
        col("player_id"),
        explode(split(col("playstyles"), ",")).alias("playstyle_raw")
    )
    .select(
        col("player_snapshot_key"),                       # preserve the composite key
        col("player_id"),                                 # preserve player_id
        trim(col("playstyle_raw")).alias("playstyle_raw") # clean the raw value
    )
    .filter(col("playstyle_raw") != "")
    .withColumn("is_plus",
        when(col("playstyle_raw").endswith("+"), True).otherwise(False)
    )
    # Extract the name without the trailing "+"
    .withColumn("playstyle_name",
        regexp_extract(col("playstyle_raw"), r"^(.+?)\+?$", 1)
    )
    .join(
        dim_playstyle.select("playstyle_id", "playstyle_name", "is_plus"),
        on=["playstyle_name", "is_plus"],
        how="left"
    )
    .select("player_snapshot_key","player_id", "playstyle_id", "is_plus")
    .distinct()
)


# ===== Create table Silver =====
spark.sql("""
CREATE TABLE IF NOT EXISTS fifa.silver.bridge_player_playstyle (
    player_snapshot_key STRING,
    player_id INT,
    playstyle_id INT,
    is_plus BOOLEAN
)
USING DELTA;
""")

# Overwrite the table
(bridge
 .write
 .mode("overwrite")
 .option("mergeSchema", "true")
 .saveAsTable("fifa.silver.bridge_player_playstyle")
)

print(f"bridge_player_playstyle creada con {spark.read.table('fifa.silver.bridge_player_playstyle').count()} filas")

LIMIT 10 of the bridge_player_playstyle table. Quick verification that we have entered the raw data into our table.

In [0]:
%sql
select * from fifa.silver.bridge_player_playstyle order by player_id limit 10;